# Predicting behavior from geometry needs a proper metric

A cohort of simulated subjects, each with a **neural geometry** and a **behavior**.
We predict behavior from geometry with three dissimilarities:

- **Procrustes** shape distance --- symmetric, a proper metric
- **linear predictivity**, $1 - R^2$ of a cross-validated linear map --- *not* symmetric
- **symmetrized predictivity**, $(D + D^\mathsf{T})/2$ --- symmetric, still not a metric

and under three ways of encoding the non-behavioral task variables: segregated,
linearly mixed, and nonlinearly (multiplicatively) mixed selectivity.

Self-contained: run top to bottom (~30 s).

In [1]:
import itertools

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from netrep.metrics import LinearMetric
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold

K, N, H = 30, 60, 4          # subjects, neurons, harmonics in the stimulus tuning
N_PCS   = 10                 # same reduction for every metric -- keeps it fair
N_THETA = 60                 # stimulus values
N_EXTRA = 5                  # other task variables in the experiment
MIX     = 0.22               # how strongly they are encoded
NOISE_VAR = 0.1              # trial-to-trial variance assumed for Fisher information

# Three types, set by how strongly each harmonic of the stimulus tuning is
# weighted: broad -> sharp.  All three span the same set of dimensions, so they
# differ by an invertible linear reweighting.
PROFILES = [np.array([1.0, 0.15, 0.05, 0.02]),
            np.array([1.0, 0.70, 0.35, 0.15]),
            np.array([1.0, 0.95, 0.85, 0.70])]

# ---- the measurement set: an ADDITIVE (one-factor-at-a-time) design.
# First the stimulus is varied with the other variables at baseline, then the
# other variables are varied at a fixed stimulus.  This is how such experiments
# are usually run -- far more realistic than a full factorial.
THETA_GRID = np.linspace(-np.pi, np.pi, N_THETA, endpoint=False)
LEVELS     = np.array(list(itertools.product([-1, 1], repeat=N_EXTRA)))
COND_THETA = np.concatenate([THETA_GRID, np.zeros(len(LEVELS))])
COND_EXTRA = np.vstack([np.zeros((N_THETA, N_EXTRA)), LEVELS])
M = len(COND_THETA)
print(f"{M} conditions = {N_THETA} stimulus + {len(LEVELS)} other-variable conditions")

92 conditions = 60 stimulus + 32 other-variable conditions


## 1. Simulate the cohort

Two things vary across subjects, independently:

1. **tuning width** --- how sharply neurons are tuned to the stimulus. This sets behavior.
2. **how many of the other task variables** the population encodes (0-5). Irrelevant to
   behavior.

We build the same cohort three times, differing only in *how* the other variables are
encoded:

| scheme | single-neuron selectivity |
|---|---|
| **segregated** | separate neuron groups: some encode the stimulus, others one variable each |
| **linear mixed** | every neuron responds to the stimulus *and* adds each variable |
| **nonlinear mixed** | the other variables multiplicatively gain-modulate the stimulus tuning |

In [2]:
def stimulus_tuning(w, mu, thetas):
    """Peak-normalised periodic tuning of each neuron to the stimulus."""
    X = sum(w[h - 1] * np.cos(h * (thetas[None, :] - mu[:, None]))
            for h in range(1, H + 1))
    return X / np.abs(X).max()


def segregated(w, n_extra, rs, n_stim=30):
    """No mixing: disjoint groups of neurons for the stimulus and each variable."""
    X = np.zeros((N, M))
    X[:n_stim] = stimulus_tuning(w, rs.uniform(-np.pi, np.pi, n_stim), COND_THETA)
    if n_extra:
        for c, g in enumerate(np.array_split(np.arange(n_stim, N), n_extra)):
            X[g] = MIX * np.outer(rs.uniform(-1, 1, len(g)), COND_EXTRA[:, c])
    return X - X.mean(axis=1, keepdims=True)


def linear_mixed(w, n_extra, rs):
    """Additive mixed selectivity: each variable adds one dimension."""
    X = stimulus_tuning(w, rs.uniform(-np.pi, np.pi, N), COND_THETA)
    for c in range(n_extra):
        X = X + MIX * np.outer(rs.uniform(-1, 1, N), COND_EXTRA[:, c])
    return X - X.mean(axis=1, keepdims=True)


def nonlinear_mixed(w, n_extra, rs):
    """Conjunctive: the other variables gain-modulate the stimulus tuning."""
    X = stimulus_tuning(w, rs.uniform(-np.pi, np.pi, N), COND_THETA)
    gain = np.ones_like(X)
    for c in range(n_extra):
        gain = gain + MIX * np.outer(rs.uniform(-1, 1, N), COND_EXTRA[:, c])
    X = X * gain
    return X - X.mean(axis=1, keepdims=True)


def acuity(w, seed, pool=2000):
    """log Fisher information per neuron: how finely the population can
    discriminate nearby stimuli.  Computed on a large independent pool, i.e. the
    asymptotic value, not from the N neurons recorded here."""
    rs = np.random.RandomState(seed)
    tc = stimulus_tuning(w, rs.uniform(-np.pi, np.pi, pool), THETA_GRID)
    d = np.gradient(tc, THETA_GRID[1] - THETA_GRID[0], axis=1)
    return np.log(np.mean(d ** 2) / NOISE_VAR)


SCHEMES = [("segregated", segregated),
           ("linear mixed", linear_mixed),
           ("nonlinear mixed", nonlinear_mixed)]

rs      = np.random.RandomState(0)
labels  = np.repeat([0, 1, 2], K // 3)                  # tuning-width type
weights = np.array([PROFILES[t] * rs.uniform(0.9, 1.1, H) for t in labels])
n_extra = rs.randint(0, N_EXTRA + 1, K)                 # how many other variables

cohorts = {name: [fn(w, int(n), np.random.RandomState(400 + i))
                  for i, (w, n) in enumerate(zip(weights, n_extra))]
           for name, fn in SCHEMES}
behavior = np.array([acuity(w, 1000 + i) for i, w in enumerate(weights)]) \
           + rs.normal(0, 0.03, K)

for name, Xs in cohorts.items():
    print(f"{name:<18} manifold rank up to {max(np.linalg.matrix_rank(X, tol=1e-8) for X in Xs)}")
print(f"behavior from {behavior.min():.2f} to {behavior.max():.2f}; "
      f"{n_extra.min()}-{n_extra.max()} other variables encoded")

segregated         manifold rank up to 13
linear mixed       manifold rank up to 13
nonlinear mixed    manifold rank up to 13
behavior from 1.25 to 2.07; 0-5 other variables encoded


## 2. The three dissimilarities

In [3]:
def d_procrustes(X, Y):
    metric = LinearMetric(alpha=1, center_columns=True, score_method="euclidean")
    metric.fit(X, Y)
    return metric.score(X, Y)


def r2_predict(X, Y):
    """Cross-validated linear predictivity of Y from X, held out over conditions."""
    return np.mean([r2_score(Y[te], RidgeCV(alphas=np.logspace(-4, 4, 17))
                             .fit(X[tr], Y[tr]).predict(X[te]),
                             multioutput="variance_weighted")
                    for tr, te in KFold(5, shuffle=True, random_state=0).split(X)])


def dissimilarities(Xs):
    P = [PCA(N_PCS).fit_transform(X.T) for X in Xs]
    Dp = np.zeros((K, K))
    Dr = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            if i < j:
                Dp[i, j] = Dp[j, i] = d_procrustes(P[i], P[j])
            if i != j:
                Dr[i, j] = 1 - max(r2_predict(P[i], P[j]), 0)
    return {"Procrustes": Dp, "predictivity, $D$": Dr,
            "predictivity, symmetrized": (Dr + Dr.T) / 2}


D = {name: dissimilarities(Xs) for name, Xs in cohorts.items()}
for name in cohorts:
    a = np.abs(D[name]["predictivity, $D$"])
    print(f"{name:<18} predictivity asymmetry "
          f"{np.abs(a - a.T).mean() / a.mean():.2f}   Procrustes "
          f"{np.abs(D[name]['Procrustes'] - D[name]['Procrustes'].T).max():.1e}")

segregated         predictivity asymmetry 1.48   Procrustes 0.0e+00
linear mixed       predictivity asymmetry 0.93   Procrustes 0.0e+00
nonlinear mixed    predictivity asymmetry 1.36   Procrustes 0.0e+00


## 3. Predict behavior from geometry

Leave-one-out **k-nearest-neighbour** regression: each subject's behavior is predicted by
averaging its 3 nearest neighbours. With an asymmetric matrix, "the neighbours of subject
*i*" is not a well-defined set.

In [4]:
def knn_predict(Dm, beh=None, k=3):
    beh = behavior if beh is None else beh
    masked = Dm + np.eye(K) * 1e9         # never let i be its own neighbour
    return np.array([beh[np.argsort(masked[i])[:k]].mean() for i in range(K)])


METRICS = ["Procrustes", "predictivity, $D$", "predictivity, symmetrized"]
print(f"{'':<28}" + "".join(f"{n:>18}" for n in cohorts))
for m in METRICS:
    row = "".join(f"{r2_score(behavior, knn_predict(D[c][m])):>+18.2f}" for c in cohorts)
    print(f"{m.replace('$','').replace(chr(92)+'mathsf{T}','T'):<28}" + row)

                                    segregated      linear mixed   nonlinear mixed
Procrustes                               +0.97             +0.98             +0.98
predictivity, D                          -0.56             -0.92             -0.51
predictivity, symmetrized                +0.30             -0.13             +0.18


## 4. Repeat over many cohorts

Everything above is one cohort. Resample the whole thing many times to see how stable
each result is.

In [5]:
from joblib import Parallel, delayed

N_SEEDS = 50


def run_seed(seed):
    rs = np.random.RandomState(seed)
    lab = np.repeat([0, 1, 2], K // 3)
    W = np.array([PROFILES[t] * rs.uniform(0.9, 1.1, H) for t in lab])
    n_ex = rs.randint(0, N_EXTRA + 1, K)
    beh = np.array([acuity(w, 10_000 + seed * K + i) for i, w in enumerate(W)]) \
          + rs.normal(0, 0.03, K)
    out = {}
    for name, fn in SCHEMES:
        Xs = [fn(w, int(n), np.random.RandomState(seed * 977 + i))
              for i, (w, n) in enumerate(zip(W, n_ex))]
        Dd = dissimilarities(Xs)
        out[name] = {m: r2_score(beh, knn_predict(Dd[m], beh)) for m in METRICS}
    return out


seeds = Parallel(n_jobs=-2)(delayed(run_seed)(s) for s in range(N_SEEDS))
R2 = {sch: {m: np.array([s[sch][m] for s in seeds]) for m in METRICS}
      for sch in cohorts}

print(f"{N_SEEDS} cohorts, mean +- sd")
print(f"{'':<28}" + "".join(f"{n:>20}" for n in cohorts))
for m in METRICS:
    lbl = m.replace('$', '')
    print(f"{lbl:<28}" + "".join(
        f"{R2[c][m].mean():>+13.2f} +-{R2[c][m].std():<5.2f}" for c in cohorts))

50 cohorts, mean +- sd
                                      segregated        linear mixed     nonlinear mixed
Procrustes                          +0.97 +-0.01         +0.97 +-0.01         +0.97 +-0.01 
predictivity, D                     -0.43 +-0.27         -0.62 +-0.23         -0.18 +-0.43 
predictivity, symmetrized           +0.33 +-0.20         -0.18 +-0.27         +0.22 +-0.20 


## 5. The figure

In [6]:
# Same figure settings as the other analysis notebooks in this repo
new_rc_params = {"text.usetex": False, "svg.fonttype": "none", "pdf.fonttype": 42,
                 "font.family": "sans-serif", "font.sans-serif": ["Arial"],
                 "mathtext.fontset": "custom", "mathtext.rm": "Arial",
                 "mathtext.it": "Arial:italic", "mathtext.bf": "Arial:bold",
                 "axes.unicode_minus": False,
                 "font.size": 8, "axes.titlesize": 8, "axes.linewidth": 0.7}
plt.rcParams.update(new_rc_params)

cmap    = sns.color_palette("husl", N_THETA)   # colour by stimulus, as in Fig. 2
rs_view = np.random.RandomState(111)           # random 3-D viewing angles, as in Fig. 2

# One colour per dissimilarity
METRIC_COLORS = {"Procrustes": "#eb6834",
                 "predictivity, $D$": "#0b0b0b",
                 "predictivity, symmetrized": "#8a8984"}
METRIC_LABELS = {"Procrustes": "Procrustes",
                 "predictivity, $D$": "predictivity $D$",
                 "predictivity, symmetrized": "$D + D^\mathsf{T}$"}

# The top rows show ONLY the behaviourally relevant conditions: the first N_THETA
# columns are the stimulus block, where the other variables sit at baseline, so
# the mixing term is exactly zero and this is pure stimulus tuning.  The linear
# mixed cohort is used because there every neuron is stimulus-tuned, so all N
# contribute.  The metrics themselves use all M conditions.
stim_block = [X[:, :N_THETA] for X in cohorts["linear mixed"]]
examples = [np.where(labels == t)[0][np.argmin(n_extra[labels == t])] for t in range(3)]
titles = ["broad tuning", "intermediate", "sharp tuning"]
lim = (behavior.min() - 0.4, behavior.max() + 0.4)
MAIN_SCHEME = "linear mixed"       # single cohort shown in the scatter

fig = plt.figure(figsize=(7.4, 6.6))
gs  = fig.add_gridspec(3, 6, height_ratios=[1.9, 0.8, 2.4], hspace=0.7, wspace=0.9)

for c in range(3):                             # one column per tuning-width type
    i = examples[c]

    ax = fig.add_subplot(gs[0, 2 * c:2 * c + 2], projection="3d")
    Q = np.linalg.qr(rs_view.randn(3, 3))[0]
    x, y, z = Q @ PCA(3).fit_transform(stim_block[i].T).T
    ax.plot(np.r_[x, x[0]], np.r_[y, y[0]], zs=np.r_[z, z[0]],
            lw=0.6, color="0.6", zorder=0)
    ax.scatter(x, y, zs=z, c=cmap, lw=0, s=12)
    ax.plot(x, y, zs=ax.get_zlim()[0], lw=3, color="k", alpha=0.3)
    ax.axis("off")
    ax.set_box_aspect(None, zoom=1.15)   # >1.2 clips the manifold
    ax.set_title(f"{titles[c]}\nbehavior = {behavior[i]:.2f}")

    ax = fig.add_subplot(gs[1, 2 * c:2 * c + 2])
    active = np.where(np.ptp(stim_block[i], axis=1) > 1e-9)[0]
    order = active[np.argsort(stim_block[i][active].argmax(axis=1))]
    for n in order[::max(len(order) // 7, 1)]:
        ax.plot(THETA_GRID, stim_block[i][n], lw=0.9,
                color=cmap[stim_block[i][n].argmax()])
    ax.set_xticks([-np.pi, 0, np.pi], [r"$-\pi$", "0", r"$\pi$"])
    ax.set_yticks([])
    ax.set_xlabel("stimulus")
    if c == 0:
        ax.set_ylabel("firing rate")
    sns.despine(ax=ax, left=True)

# --- bottom left: behaviour predicted from geometry, one cohort
ax = fig.add_subplot(gs[2, 0:3])
ax.plot(lim, lim, "-", lw=0.7, color="0.7", zorder=0)
for k, metric in enumerate(reversed(METRICS)):     # Procrustes drawn last, on top
    pred = knn_predict(D[MAIN_SCHEME][metric])
    ax.scatter(behavior, pred, s=13, edgecolors="none", color=METRIC_COLORS[metric])
    ax.text(0.02, 0.97 - 0.085 * k,
            f"{METRIC_LABELS[metric]} ($R^2$ = {r2_score(behavior, pred):.2f})",
            transform=ax.transAxes, va="top", fontsize=6.5,
            color=METRIC_COLORS[metric])
ax.set(xlim=lim, ylim=lim)
ax.set_aspect("equal")
ax.set_xlabel("true behavior")
ax.set_ylabel("predicted (kNN, $k$=3)")
ax.set_title("one cohort")
sns.despine(ax=ax)

# --- bottom right: across cohorts
ax = fig.add_subplot(gs[2, 3:6])
for si, scheme in enumerate(cohorts):
    for mi, metric in enumerate(METRICS):
        vals = R2[scheme][metric]
        pos = si * 4 + mi + 1
        body = ax.violinplot([vals], positions=[pos], showextrema=False,
                             widths=0.85)["bodies"][0]
        body.set_facecolor(METRIC_COLORS[metric])
        ax.plot([pos - 0.38, pos + 0.38], [vals.mean()] * 2, "-", lw=1.6,
                color=METRIC_COLORS[metric])
ax.axhline(0, ls=":", lw=0.7, color="0.4")
ax.set_xticks([2, 6, 10], ["segregated", "linear\nmixed", "nonlinear\nmixed"])
ax.set_xlim(0, 12)
ax.set_ylabel("$R^2$")
ax.set_box_aspect(1)               # square panel
ax.set_title(f"{N_SEEDS} cohorts")
ax.legend(handles=[plt.Line2D([], [], lw=4, color=METRIC_COLORS[m],
                              label=METRIC_LABELS[m]) for m in METRICS],
          fontsize=6, frameon=False, loc="lower left", handlelength=1.2,
          handletextpad=0.4, labelspacing=0.2)
sns.despine(ax=ax)

fig.savefig("figure_metric_vs_regression.pdf", transparent=True, bbox_inches="tight")
fig.savefig("figure_metric_vs_regression.svg", transparent=True, bbox_inches="tight")
plt.show()

findfont: Failed to find font weight normal, now using 0.


/var/folders/6w/snfd5vc50jb775fq8364m1d80000gn/T/ipykernel_37870/895639307.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
